# Ejercicio 1. C++

Se resolvió todo el ejercico en un único archivo .cpp

Como en el fenomeno simulado el tiempo es importante, se buscó tener un paso del tiempo coordinado para todos los hilos. Para eso se utilizo una barrera y la granularidad fueron las horas.

In [ ]:
%%writefile simulation_trucks.cpp

#include <atomic>
#include <barrier>
#include <cassert>
#include <functional>
#include <iostream>
#include <memory>
#include <mutex>
#include <semaphore>
#include <string>
#include <thread>
#include <utility>

const int SIMULATION_LIMIT = 1e9 + 1;

std::pair<int, int> process_args(int argc, char *argv[])
{
  try
  {
    if (argc != 3)
    {
      throw std::invalid_argument("Wrong number of arguments");
    }

    int num_trucks = std::stoi(std::string(argv[1]));
    int num_travels = std::stoi(std::string(argv[2]));

    if (num_trucks <= 0 || num_travels <= 0 || num_trucks > SIMULATION_LIMIT || num_travels > SIMULATION_LIMIT)
    {
      throw std::invalid_argument("The number of trucks and travels must be positive integers up to " + std::to_string(SIMULATION_LIMIT));
    }

    return std::make_pair(num_trucks, num_travels);
  }
  catch (const std::exception &e)
  {
    std::cerr << "Error: " << e.what() << std::endl;
    std::cerr << "Usage: " << argv[0] << " <num_trucks> <num_travels>" << std::endl;
    exit(1);
  }
  assert(false);
}

const int MIN_TRAVEL_TIME = 18, MAX_TRAVEL_TIME = 24;
const int GAS_LOAD_TIME = 1, FERNANDEZ_GAS_STATIONS = 2;
const int LOAD_TIME = 2, UNLOAD_TIME = 2;

std::counting_semaphore<SIMULATION_LIMIT> tapiales_travels(0), fernandez_travels(0);
std::binary_semaphore tapiales_load(1), fernandez_load(1), tapiales_unload(1), fernandez_unload(1);
std::counting_semaphore<FERNANDEZ_GAS_STATIONS> fernandez_gas_station(FERNANDEZ_GAS_STATIONS);
std::mutex cout_mutex;

std::atomic<int> hours_passed(0);

void advance_hour()
{
  hours_passed++;
  std::lock_guard<std::mutex> cout_lock(cout_mutex);
  std::cout << std::string(50, '=');
  std::cout << "[HOUR " << hours_passed << "]";
  std::cout << std::string(50, '=') << std::endl;
}

using CompletitionFunction = std::function<void()>;

std::shared_ptr<std::barrier<CompletitionFunction>> clock_barrier = nullptr;

void simulate_time_passage(int hours_to_pass, std::string message = "")
{
  for (int i = 0; i < hours_to_pass; i++)
  {
    cout_mutex.lock();
    std::cout << message << std::endl;
    cout_mutex.unlock();

    clock_barrier->arrive_and_wait();
  }
}

void load_in_tapiales(int truck_id)
{
  while (!tapiales_load.try_acquire())
  {
    simulate_time_passage(1, "Truck " + std::to_string(truck_id) + " is waiting to load in Tapiales");
  }
  simulate_time_passage(LOAD_TIME, "Truck " + std::to_string(truck_id) + " is loading in Tapiales");
  tapiales_load.release();
}

void travel_from_tapiales_to_fernandez(int truck_id)
{
  int travel_time = rand() % (MAX_TRAVEL_TIME - MIN_TRAVEL_TIME + 1) + MIN_TRAVEL_TIME;
  simulate_time_passage(travel_time, "Truck " + std::to_string(truck_id) + " is traveling from Tapiales to Fernandez");
}

void unload_in_fernandez(int truck_id)
{
  while (!fernandez_unload.try_acquire())
  {
    simulate_time_passage(1, "Truck " + std::to_string(truck_id) + " is waiting to unload in Fernandez");
  }
  simulate_time_passage(UNLOAD_TIME, "Truck " + std::to_string(truck_id) + " is unloading in Fernandez");
  fernandez_unload.release();
}

void tapiales_to_fernandez(int truck_id)
{
  load_in_tapiales(truck_id);
  travel_from_tapiales_to_fernandez(truck_id);
  unload_in_fernandez(truck_id);
}

void load_in_fernandez(int truck_id)
{
  while (!fernandez_load.try_acquire())
  {
    simulate_time_passage(1, "Truck " + std::to_string(truck_id) + " is waiting to load in Fernandez");
  }
  simulate_time_passage(LOAD_TIME, "Truck " + std::to_string(truck_id) + " is loading in Fernandez");
  fernandez_load.release();
}

void load_gas_in_fernandez(int truck_id)
{
  while (!fernandez_gas_station.try_acquire())
  {
    simulate_time_passage(1, "Truck " + std::to_string(truck_id) + " is waiting to load gas in Fernandez");
  }
  simulate_time_passage(GAS_LOAD_TIME, "Truck " + std::to_string(truck_id) + " is loading gas in Fernandez");
  fernandez_gas_station.release();
}

void travel_from_fernandez_to_tapiales(int truck_id)
{
  int travel_time = rand() % (MAX_TRAVEL_TIME - MIN_TRAVEL_TIME + 1) + MIN_TRAVEL_TIME;
  simulate_time_passage(travel_time, "Truck " + std::to_string(truck_id) + " is traveling from Fernandez to Tapiales");
}

void unload_in_tapiales(int truck_id)
{
  while (!tapiales_unload.try_acquire())
  {
    simulate_time_passage(1, "Truck " + std::to_string(truck_id) + " is waiting to unload in Tapiales");
  }
  simulate_time_passage(UNLOAD_TIME, "Truck " + std::to_string(truck_id) + " is unloading in Tapiales");
  tapiales_unload.release();
}

void fernandez_to_tapiales(int truck_id)
{
  load_in_fernandez(truck_id);
  load_gas_in_fernandez(truck_id);
  travel_from_fernandez_to_tapiales(truck_id);
  unload_in_tapiales(truck_id);
}

void truck_simulation_core(int truck_id)
{
  while (true)
  {
    if (!tapiales_travels.try_acquire())
    {
      break;
    }
    tapiales_to_fernandez(truck_id);

    if (!fernandez_travels.try_acquire())
    {
      break;
    }

    fernandez_to_tapiales(truck_id);
  }
}

int truck_simulation(int truck_id)
{
  cout_mutex.lock();
  std::cout << "Truck " << truck_id << " is starting its travels" << std::endl;
  cout_mutex.unlock();

  clock_barrier->arrive_and_wait();

  truck_simulation_core(truck_id);

  cout_mutex.lock();
  std::cout << "Truck " << truck_id << " has finished its travels" << std::endl;
  cout_mutex.unlock();

  clock_barrier->arrive_and_drop();

  return hours_passed;
}

void run_main_simulation(int num_trucks)
{
  std::thread trucks[num_trucks];
  for (int truck_id = 1; truck_id <= num_trucks; truck_id++)
  {
    trucks[truck_id - 1] = std::thread(truck_simulation, truck_id);
  }

  for (int i = 0; i < num_trucks; i++)
  {
    trucks[i].join();
  }

  std::cout << "Simulation finished" << std::endl;
}

int main_simulation(int num_trucks, int num_travels)
{
  std::cout << "Starting simulation with " << num_trucks << " trucks and " << num_travels << " travels" << std::endl;
  tapiales_travels.release(num_travels);
  fernandez_travels.release(num_travels);

  clock_barrier = std::make_shared<std::barrier<CompletitionFunction>>(num_trucks, advance_hour);

  run_main_simulation(num_trucks);

  std::cout << "All trucks have finished their travels in " << hours_passed << " hours " << std::endl;
  const int HOURS_IN_A_DAY = 24;
  std::cout << "This is " << hours_passed / HOURS_IN_A_DAY << " days and " << hours_passed % HOURS_IN_A_DAY << " hours " << std::endl;
  return 0;
}

int main(int argc, char *argv[])
{
  auto [num_trucks, num_travels] = process_args(argc, argv);
  main_simulation(num_trucks, num_travels);
  return 0;
}


Writing simulation_trucks.cpp




---

**Compilación**

Se compila el programa.

In [ ]:
!g++ simulation_trucks.cpp -O2 -std=c++20 -o simulation_trucks



---

**Ejecución**

Se ejecuta el programa para 8 camiones y 12 viajes.

In [ ]:
!./simulation_trucks 8 12

Starting simulation with 8 trucks and 12 travels
Truck 1 is starting its travels
Truck 2 is starting its travels
Truck 3 is starting its travels
Truck 4 is starting its travels
Truck 5 is starting its travels
Truck 6 is starting its travels
Truck 7 is starting its travels
Truck 8 is starting its travels
==================================================[HOUR 1]==================================================
Truck 1 is loading in Tapiales
Truck 8 is waiting to load in Tapiales
Truck 7 is waiting to load in Tapiales
Truck 4 is waiting to load in Tapiales
Truck 5 is waiting to load in Tapiales
Truck 3 is waiting to load in Tapiales
Truck 2 is waiting to load in Tapiales
Truck 6 is waiting to load in Tapiales
==================================================[HOUR 2]==================================================
Truck 1 is loading in Tapiales
Truck 3 is waiting to load in Tapiales
Truck 2 is waiting to load in Tapiales
Truck 7 is waiting to load in Tapiales
Truck 4 is waiting to loa

# Ejercicio 2. Java

Se creo un archivo GiorgiosBakery.java que contiene todas las clases necesarias para representar a la panaderia de Giorgio. Se divide en:


*   Kneader - Maestro y Asistente que amasaon los bollos
*   Baker - Encargado de cocinar el Pan
*   Packer - 2 Encargados empaquetar el pan y ponerlo en la canasta
*   Cliente - N clientes que compran pan
*   Vendedor - Se toma el dato en la clase cliente cuando realizan la "Compra"







In [ ]:
%%writefile GiorgiosBakery.java

import java.util.concurrent.*;
import java.util.*;

public class GiorgiosBakery
{

  private static final int TABLE_CAPACITY = 20;
  private static final int BASKET_CAPACITY = 50;
  private static final int COUNTER_CAPACITY = 30;

  private static final BlockingQueue<String> doughTable = new ArrayBlockingQueue<>(TABLE_CAPACITY);
  private static final BlockingQueue<String> breadBasket = new ArrayBlockingQueue<>(BASKET_CAPACITY);
  private static final BlockingQueue<String> counter = new ArrayBlockingQueue<>(COUNTER_CAPACITY);

  private static final Semaphore scale = new Semaphore(1);
  private static final Semaphore labelMachines = new Semaphore(2);

  private static final Random random = new Random();
  private static int totalSales = 0;
  private static final Object salesLock = new Object();

  private static int totalClients;
  private static final Object clientLock = new Object();
  private static int clientsServed = 0;

  private static final Object doughTableLock = new Object();

  private static final String[] names = {"Alice", "Bob", "Charlie", "Diana", "Eve", "Frank"};

  public static void main(String[] args)
  {
    if (args.length != 1)
    {
      System.out.println("Usage: java GiorgiosBakery <number_of_clients>");
      return;
    }

    totalClients = Integer.parseInt(args[0]);

    Thread master = new Thread(new Kneader("Master", 2));
    Thread assistant = new Thread(new Kneader("Assistant", 1));
    Thread baker = new Thread(new Baker());
    Thread packer1 = new Thread(new Packer("Packer 1"));
    Thread packer2 = new Thread(new Packer("Packer 2"));

    master.start();
    assistant.start();
    baker.start();
    packer1.start();
    packer2.start();

    List<Thread> clientThreads = new ArrayList<>();
    for (int i = 1; i <= totalClients; i++)
    {
      Thread client = new Thread(new Client(i));
      client.start();
      clientThreads.add(client);
    }

    for (Thread client : clientThreads)
    {
      try
      {
        client.join();
      } catch (InterruptedException e)
      {
        Thread.currentThread().interrupt();
      }
    }

    System.out.println("All clients served. Bakery is closing.");
    System.exit(0);
  }

  static class Kneader implements Runnable
  {
    private final String name;
    private final int doughBallsPerTurn;

    public Kneader(String name, int doughBallsPerTurn)
    {
      this.name = name;
      this.doughBallsPerTurn = doughBallsPerTurn;
    }

    @Override
    public void run()
    {
      try
      {
        while (true)
        {
          synchronized (clientLock)
          {
            if (clientsServed >= totalClients) break;
          }
          System.out.println(name + " is kneading: " + doughBallsPerTurn + " dough ball(s)");
          Thread.sleep(5000);

          synchronized (doughTableLock)
          {
            while (doughTable.size() > TABLE_CAPACITY - this.doughBallsPerTurn) {
              doughTableLock.wait();
            }

            for (int i = 0; i < doughBallsPerTurn; i++)
            {
              doughTable.put("Dough Ball");
              doughTableLock.notifyAll();
            }
            System.out.println(name + " put " + doughBallsPerTurn + " dough ball(s). Balls in Table: " + doughTable.size());
          }
        }
      } catch (InterruptedException e)
      {
        Thread.currentThread().interrupt();
      }
    }
  }

  static class Baker implements Runnable
  {
    @Override
    public void run()
    {
      try
      {
        while (true)
        {
          synchronized (clientLock)
          {
            if (clientsServed >= totalClients && doughTable.size() < 5) break;
          }
          synchronized (doughTableLock)
          {
            while (doughTable.size() < 5)
            {
              doughTableLock.wait();
            }
            List<String> batch = new ArrayList<>();
            for (int i = 0; i < 5; i++) {
              batch.add(doughTable.take());
            }
            System.out.println("Baker took 5 dough balls. Remaining in Table: " + doughTable.size());
            Thread.sleep(1000);
            doughTableLock.notifyAll();
          }
          System.out.println("Baker baking batch of 5 dough balls...");
          Thread.sleep(10000);

          synchronized (breadBasket)
          {
            while (breadBasket.remainingCapacity() < 5)
            {
              System.out.println("Oven blocked. Waiting for basket space...");
              breadBasket.wait();
            }
            for (int i = 0; i < 5; i++)
            {
              breadBasket.put("Bread");
            }
            System.out.println("Batch baked. Basket: " + breadBasket.size());
            breadBasket.notifyAll();
          }
        }
      } catch (InterruptedException e)
      {
        Thread.currentThread().interrupt();
      }
    }
  }

  static class Packer implements Runnable
  {
    private final String name;

    public Packer(String name)
    {
      this.name = name;
    }

    @Override
    public void run()
    {
      try
      {
        while (true)
        {
          synchronized (clientLock)
          {
            if (clientsServed >= totalClients && breadBasket.size() < 3) break;
          }

          List<String> packageItems = new ArrayList<>();
          for (int i = 0; i < 3; i++)
          {
            packageItems.add(breadBasket.take());
          }
          System.out.println(name + " took 3 breads. Remaining in Basket: " + breadBasket.size());

          synchronized (breadBasket)
          {
            breadBasket.notifyAll();
          }

          scale.acquire();
          System.out.println(name + " weighing package...");
          Thread.sleep(1000);
          scale.release();

          labelMachines.acquire();
          System.out.println(name + " labeling package...");
          Thread.sleep(1000);
          labelMachines.release();

          counter.put("Package");
          System.out.println(name + " placed package. Counter: " + counter.size());
        }
      } catch (InterruptedException e) {
        Thread.currentThread().interrupt();
      }
    }
  }

  static class Client implements Runnable
  {
    private final int id;
    private final String name;

    public Client(int id)
    {
      this.id = id;
      this.name = names[random.nextInt(names.length)];
    }

    @Override
    public void run()
    {
      try {
        int packages = 1 + random.nextInt(3);
        for (int i = 0; i < packages; i++)
        {
          counter.take();
          System.out.println("Client " + id + " (" + name + ") took 1 package. Remaining in Counter: " + counter.size());
        }
        synchronized (salesLock)
        {
          totalSales += packages;
        }
        synchronized (clientLock)
        {
          clientsServed++;
        }
        System.out.println("Client " + id + " (" + name + ") bought " + packages + " package(s). Total sales: " + totalSales);
      } catch (InterruptedException e) {
        Thread.currentThread().interrupt();
      }
    }
  }
}


Overwriting GiorgiosBakery.java




---

**Compilación**

Se compila la clase:

In [ ]:
!javac GiorgiosBakery.java

---

**Ejecución**

Se ejecuta el el proceso de panaderia ingresando la cantidad de clientes:

In [ ]:
!java GiorgiosBakery 2

Master is kneading: 2 dough ball(s)
Assistant is kneading: 1 dough ball(s)
Master put 2 dough ball(s). Balls in Table: 2
Assistant put 1 dough ball(s). Balls in Table: 3
Assistant is kneading: 1 dough ball(s)
Master is kneading: 2 dough ball(s)
Assistant put 1 dough ball(s). Balls in Table: 4
Assistant is kneading: 1 dough ball(s)
Master put 2 dough ball(s). Balls in Table: 6
Master is kneading: 2 dough ball(s)
Baker took 5 dough balls. Remaining in Table: 1
Baker baking batch of 5 dough balls...
Assistant put 1 dough ball(s). Balls in Table: 2
Assistant is kneading: 1 dough ball(s)
Master put 2 dough ball(s). Balls in Table: 4
Master is kneading: 2 dough ball(s)
Assistant put 1 dough ball(s). Balls in Table: 5
Assistant is kneading: 1 dough ball(s)
Master put 2 dough ball(s). Balls in Table: 7
Master is kneading: 2 dough ball(s)
Batch baked. Basket: 5
Baker took 5 dough balls. Remaining in Table: 2
Packer 2 took 3 breads. Remaining in Basket: 2
Packer 2 weighing package...
Baker bakin

# Ejercicio 3. Python

**Código**

A continuación se presenta la implementación del programa en Python, cuyo objetivo es reflejar el funcionamiento del popular taller de restauración de Richard y Aaron.

In [ ]:
%%writefile gas_monkey.py
import threading
import time
import random
import sys

# Constants
PARKING_LOT_CAPACITY = 6
PARKING_LOT_ENTRY_CONTROL = 5
PIT_CAPACITY = 3
SERVICE_CAPACITY = 2
NUM_ASSISTANTS = 2

# Semaphores & Locks
parking_lot = threading.BoundedSemaphore(PARKING_LOT_CAPACITY)
parking_lot_entry = threading.BoundedSemaphore(PARKING_LOT_ENTRY_CONTROL)
main_lane = threading.Semaphore(1)
pit_lane = threading.Semaphore(1)
pit_area = threading.BoundedSemaphore(PIT_CAPACITY)
service_area = threading.BoundedSemaphore(SERVICE_CAPACITY)
assistants = threading.Semaphore(NUM_ASSISTANTS)
aaron_lock = threading.Lock()
charles_lock = threading.Lock()

class Car(threading.Thread):

  def __init__(self, car_id):
    super().__init__()
    self.car_id = car_id

  def run(self):
    try:
      self.arrive_and_park()
      self.inspect()
      self.transfer_to_pit()
      self.repair()
      self.transfer_to_service()
      self.oil_change()
      self.transfer_to_wash()
      self.wash()
      self.exit_shop()
    except Exception as e:
      print(f"Car {self.car_id} error: {e}")

  def arrive_and_park(self):
    print(f"Car {self.car_id}: Arrived, waiting for parking...")
    parking_lot_entry.acquire()
    parking_lot.acquire()
    main_lane.acquire()
    print(f"Car {self.car_id}: Entering lot.")
    time.sleep(random.uniform(0.1, 0.3))
    main_lane.release()

  def inspect(self):
    with aaron_lock:
      print(f"Car {self.car_id}: Inspecting by Aaron.")
      time.sleep(random.uniform(0.5, 1.0))

  def transfer_to_pit(self):
    pit_lane.acquire()
    pit_area.acquire()
    assistants.acquire()
    print(f"Car {self.car_id}: Moved to pit.")
    parking_lot.release()
    parking_lot_entry.release()
    assistants.release()
    time.sleep(random.uniform(0.1, 0.3))
    pit_lane.release()

  def repair(self):
    with charles_lock:
      print(f"Car {self.car_id}: Repair by Charles.")
      time.sleep(random.uniform(1.0, 2.0))

  def transfer_to_service(self):
    assistants.acquire()
    assistants.acquire()
    service_area.acquire()
    print(f"Car {self.car_id}: Moved to service.")
    pit_area.release()
    time.sleep(random.uniform(0.1, 0.3))

  def oil_change(self):
    print(f"Car {self.car_id}: Oil change.")
    time.sleep(random.uniform(0.5, 1.0))

  def transfer_to_wash(self):
    parking_lot.acquire()
    print(f"Car {self.car_id}: Heading to wash.")
    assistants.release()
    assistants.release()
    time.sleep(random.uniform(0.1, 0.3))

  def wash(self):
    print(f"Car {self.car_id}: Washing.")
    time.sleep(random.uniform(0.5, 1.5))
    service_area.release()

  def exit_shop(self):
    main_lane.acquire()
    print(f"Car {self.car_id}: Exiting shop.")
    time.sleep(random.uniform(0.1, 0.3))
    main_lane.release()
    parking_lot.release()
    print(f"Car {self.car_id}: Picked up.")

def simulate(n):
  cars = []
  for i in range(n):
    car = Car(i+1)
    cars.append(car)
    car.start()
  for c in cars:
    c.join()

if __name__ == '__main__':
  if len(sys.argv) != 2:
    print("Usage: python gas_monkey.py <num_cars>")
    sys.exit(1)
  simulate(int(sys.argv[1]))

Overwriting gas_monkey.py




---

**Consideración importante**

Se asumió que **el lavado forma parte del área de servicio**, esto le da sentido a que "los dos ayudantes novatos se encargarán de mover el auto reparado desde la zona de fosas hasta el área de servicio (capacidad 2 autos) y realizar el cambio de aceite". Si el lavado no formara parte del área de servicio, y si desde que el vehículo sale de la zona de fosas hasta que comienza el lavado siempre es acompañado por ambos ayudantes, no tendría sentido que el área de servicio tenga capacidad para 2 autos, ya que siempre habría uno sólo.



---

**Ejecución**

Se ejecuta el programa para una cantidad de 20 clientes (autos).

In [ ]:
!python3 gas_monkey.py 20

Car 1: Arrived, waiting for parking...
Car 1: Entering lot.
Car 2: Arrived, waiting for parking...
Car 3: Arrived, waiting for parking...
Car 4: Arrived, waiting for parking...
Car 5: Arrived, waiting for parking...
Car 6: Arrived, waiting for parking...
Car 7: Arrived, waiting for parking...
Car 8: Arrived, waiting for parking...
Car 9: Arrived, waiting for parking...
Car 10: Arrived, waiting for parking...
Car 11: Arrived, waiting for parking...
Car 12: Arrived, waiting for parking...
Car 13: Arrived, waiting for parking...
Car 14: Arrived, waiting for parking...
Car 15: Arrived, waiting for parking...
Car 16: Arrived, waiting for parking...
Car 17: Arrived, waiting for parking...
Car 18: Arrived, waiting for parking...
Car 19: Arrived, waiting for parking...
Car 20: Arrived, waiting for parking...
Car 1: Inspecting by Aaron.
Car 2: Entering lot.
Car 3: Entering lot.
Car 4: Entering lot.
Car 5: Entering lot.
Car 1: Moved to pit.
Car 2: Inspecting by Aaron.
Car 6: Entering lot.
Car 1: